# 🐟 Grupo 7 – La Memoria de Pez
## Taller: Deep Learning Audit – El Rescate de HealthTech
**Universidad Privada del Norte – Escuela de Posgrado**

---

### Sabotaje asignado
El modelo tiene un **dataset pequeño** pero **no usa Regularización ni Weight Decay**, sufriendo un **sobreajuste (overfitting) violento**.

### Objetivo
1. Demostrar el overfitting con el modelo saboteado.
2. Diagnosticar la causa raíz.
3. Aplicar la corrección (Regularización + Weight Decay).
4. Comparar resultados Antes vs Después.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, random_split
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Reproducibilidad
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

## 1. Preparación del Dataset (pequeño)
Usamos el dataset de cáncer de mama (569 muestras, 30 features) y tomamos solo una **pequeña fracción** para simular un dataset limitado, como ocurre en imágenes médicas.

In [ ]:
# Cargar dataset de patologías (cáncer de mama)
data = load_breast_cancer()
X = data.data
y = data.target

# Tomar SOLO 80 muestras para simular dataset pequeño
indices = np.random.choice(len(X), size=80, replace=False)
X_small = X[indices]
y_small = y[indices]

# Escalar
scaler = StandardScaler()
X_small = scaler.fit_transform(X_small)

# Convertir a tensores
X_tensor = torch.FloatTensor(X_small).to(device)
y_tensor = torch.FloatTensor(y_small).unsqueeze(1).to(device)

# Split: 50 train, 30 test (proporción realística en medicina)
dataset = TensorDataset(X_tensor, y_tensor)
train_dataset, test_dataset = random_split(dataset, [50, 30])

train_loader = DataLoader(train_dataset, batch_size=10, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=30)

print(f'Dataset total: {len(X_small)} muestras')
print(f'Entrenamiento: {len(train_dataset)} muestras')
print(f'Test: {len(test_dataset)} muestras')
print(f'Features: {X_small.shape[1]}')
print(f'\n⚠️ Dataset MUY pequeño – alto riesgo de overfitting')

## 2. Modelo SABOTEADO (Sin Regularización ni Decay)
Red sobredimensionada para el tamaño del dataset, **sin ninguna técnica de regularización**.

In [ ]:
class ModeloSaboteado(nn.Module):
    """Red sobredimensionada SIN regularización.
    Con 50 muestras y ~30K parámetros, memorizará el training set."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(30, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

# Contar parámetros
modelo_saboteado = ModeloSaboteado().to(device)
n_params = sum(p.numel() for p in modelo_saboteado.parameters())
print(f'Parámetros del modelo: {n_params:,}')
print(f'Muestras de entrenamiento: 50')
print(f'Ratio parámetros/muestras: {n_params/50:.0f}:1')
print(f'\n❌ Ratio >> 1 = El modelo puede MEMORIZAR todo el dataset')

In [ ]:
def entrenar_modelo(model, train_loader, test_loader, epochs=300, lr=0.001, weight_decay=0.0):
    """Entrena y registra métricas de train y test."""
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    history = {'train_loss': [], 'test_loss': [], 'train_acc': [], 'test_acc': []}

    for epoch in range(epochs):
        # --- Train ---
        model.train()
        train_loss_total = 0
        train_correct = 0
        train_total = 0
        for X_batch, y_batch in train_loader:
            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()
            train_loss_total += loss.item() * len(y_batch)
            train_correct += ((outputs > 0.5).float() == y_batch).sum().item()
            train_total += len(y_batch)

        # --- Test ---
        model.eval()
        test_loss_total = 0
        test_correct = 0
        test_total = 0
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                outputs = model(X_batch)
                loss = criterion(outputs, y_batch)
                test_loss_total += loss.item() * len(y_batch)
                test_correct += ((outputs > 0.5).float() == y_batch).sum().item()
                test_total += len(y_batch)

        history['train_loss'].append(train_loss_total / train_total)
        history['test_loss'].append(test_loss_total / test_total)
        history['train_acc'].append(train_correct / train_total)
        history['test_acc'].append(test_correct / test_total)

        if (epoch + 1) % 50 == 0:
            print(f'Epoch {epoch+1:3d} | '
                  f'Train Loss: {history["train_loss"][-1]:.4f} | '
                  f'Test Loss: {history["test_loss"][-1]:.4f} | '
                  f'Train Acc: {history["train_acc"][-1]:.2%} | '
                  f'Test Acc: {history["test_acc"][-1]:.2%}')

    return history

In [ ]:
print('=' * 70)
print('❌ ENTRENANDO MODELO SABOTEADO (Sin Regularización ni Weight Decay)')
print('=' * 70)

# SIN weight_decay (el sabotaje)
history_saboteado = entrenar_modelo(
    modelo_saboteado, train_loader, test_loader,
    epochs=300, lr=0.001, weight_decay=0.0  # ← SIN DECAY
)

## 3. Diagnóstico del Overfitting
Visualizamos la brecha entre train y test (la "memoria de pez" del modelo).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pérdida
axes[0].plot(history_saboteado['train_loss'], label='Train Loss', color='blue')
axes[0].plot(history_saboteado['test_loss'], label='Test Loss', color='red')
axes[0].set_title('❌ Modelo Saboteado – Pérdida', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCE Loss')
axes[0].legend(fontsize=12)
axes[0].annotate('OVERFITTING\n(brecha creciente)',
                 xy=(200, history_saboteado['test_loss'][200]),
                 fontsize=11, color='red', fontweight='bold')

# Accuracy
axes[1].plot(history_saboteado['train_acc'], label='Train Acc', color='blue')
axes[1].plot(history_saboteado['test_acc'], label='Test Acc', color='red')
axes[1].set_title('❌ Modelo Saboteado – Accuracy', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].legend(fontsize=12)
axes[1].set_ylim([0.4, 1.05])

plt.tight_layout()
plt.savefig('diagnostico_overfitting.png', dpi=150, bbox_inches='tight')
plt.show()

gap = history_saboteado['train_acc'][-1] - history_saboteado['test_acc'][-1]
print(f'\n📊 DIAGNÓSTICO:')
print(f'   Train Accuracy final: {history_saboteado["train_acc"][-1]:.2%}')
print(f'   Test Accuracy final:  {history_saboteado["test_acc"][-1]:.2%}')
print(f'   Brecha (Gap):         {gap:.2%}')
print(f'\n❌ Síntoma: Train accuracy ~100% pero Test accuracy mucho menor.')
print(f'   El modelo MEMORIZÓ los datos en vez de aprender patrones.')

## 4. Análisis de Pesos (Evidencia del Overfitting)
Cuando no hay regularización, los pesos crecen sin control.

In [ ]:
# Visualizar distribución de pesos del modelo saboteado
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

all_weights = []
for name, param in modelo_saboteado.named_parameters():
    if 'weight' in name:
        all_weights.extend(param.data.cpu().numpy().flatten())

axes[0].hist(all_weights, bins=80, color='red', alpha=0.7, edgecolor='black')
axes[0].set_title('❌ Pesos del Modelo Saboteado', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Valor del peso')
axes[0].set_ylabel('Frecuencia')
axes[0].axvline(x=0, color='black', linestyle='--')

print(f'Estadísticas de pesos (Modelo Saboteado):')
print(f'  Media:  {np.mean(all_weights):.4f}')
print(f'  Std:    {np.std(all_weights):.4f}')
print(f'  Máximo: {np.max(np.abs(all_weights)):.4f}')
print(f'  ❌ Pesos grandes = modelo memorizando ruido')

## 5. ✅ CORRECCIÓN: Modelo con Regularización + Weight Decay + Dropout

### Interruptores aplicados:
1. **Weight Decay (L2)** en el optimizador Adam (`weight_decay=0.01`)
2. **Dropout** entre capas (`p=0.3`)
3. **Reducción de capacidad** (menos neuronas)
4. **Early Stopping** implícito (entrenamos menos epochs)

In [ ]:
class ModeloCorregido(nn.Module):
    """Red con Dropout + arquitectura más adecuada al tamaño del dataset."""
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(30, 64),
            nn.ReLU(),
            nn.Dropout(0.3),       # ← REGULARIZACIÓN: Dropout
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.3),       # ← REGULARIZACIÓN: Dropout
            nn.Linear(32, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

modelo_corregido = ModeloCorregido().to(device)
n_params_corr = sum(p.numel() for p in modelo_corregido.parameters())
print(f'Parámetros modelo corregido: {n_params_corr:,} (vs {n_params:,} saboteado)')
print(f'Reducción: {(1 - n_params_corr/n_params)*100:.1f}%')

In [ ]:
print('=' * 70)
print('✅ ENTRENANDO MODELO CORREGIDO (Con Regularización + Weight Decay)')
print('=' * 70)

# CON weight_decay=0.01 (la corrección clave)
history_corregido = entrenar_modelo(
    modelo_corregido, train_loader, test_loader,
    epochs=300, lr=0.001, weight_decay=0.01  # ← WEIGHT DECAY activado
)

## 6. Comparación: Antes vs Después

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# --- Fila 1: SABOTEADO ---
axes[0, 0].plot(history_saboteado['train_loss'], label='Train', color='blue')
axes[0, 0].plot(history_saboteado['test_loss'], label='Test', color='red')
axes[0, 0].set_title('❌ SABOTEADO – Pérdida', fontsize=13, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].set_ylabel('Loss')

axes[0, 1].plot(history_saboteado['train_acc'], label='Train', color='blue')
axes[0, 1].plot(history_saboteado['test_acc'], label='Test', color='red')
axes[0, 1].set_title('❌ SABOTEADO – Accuracy', fontsize=13, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].set_ylim([0.4, 1.05])

# --- Fila 2: CORREGIDO ---
axes[1, 0].plot(history_corregido['train_loss'], label='Train', color='blue')
axes[1, 0].plot(history_corregido['test_loss'], label='Test', color='green')
axes[1, 0].set_title('✅ CORREGIDO – Pérdida', fontsize=13, fontweight='bold')
axes[1, 0].legend()
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')

axes[1, 1].plot(history_corregido['train_acc'], label='Train', color='blue')
axes[1, 1].plot(history_corregido['test_acc'], label='Test', color='green')
axes[1, 1].set_title('✅ CORREGIDO – Accuracy', fontsize=13, fontweight='bold')
axes[1, 1].legend()
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Accuracy')
axes[1, 1].set_ylim([0.4, 1.05])

plt.suptitle('ANTES vs DESPUÉS de la Intervención del Equipo', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('antes_vs_despues.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Comparación de pesos
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

weights_sab = []
for name, param in modelo_saboteado.named_parameters():
    if 'weight' in name:
        weights_sab.extend(param.data.cpu().numpy().flatten())

weights_corr = []
for name, param in modelo_corregido.named_parameters():
    if 'weight' in name:
        weights_corr.extend(param.data.cpu().numpy().flatten())

axes[0].hist(weights_sab, bins=80, color='red', alpha=0.7, edgecolor='black')
axes[0].set_title('❌ Pesos – Sin Regularización', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Valor del peso')
axes[0].axvline(x=0, color='black', linestyle='--')

axes[1].hist(weights_corr, bins=80, color='green', alpha=0.7, edgecolor='black')
axes[1].set_title('✅ Pesos – Con Regularización', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Valor del peso')
axes[1].axvline(x=0, color='black', linestyle='--')

plt.tight_layout()
plt.savefig('comparacion_pesos.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Pesos saboteado  – Std: {np.std(weights_sab):.4f}, Max: {np.max(np.abs(weights_sab)):.4f}')
print(f'Pesos corregido  – Std: {np.std(weights_corr):.4f}, Max: {np.max(np.abs(weights_corr)):.4f}')

## 7. Tabla Resumen Final

In [ ]:
print('\n' + '=' * 70)
print('                    RESUMEN DE AUDITORÍA – GRUPO 7')
print('=' * 70)
print(f'{"Métrica":<30} {"Saboteado":>15} {"Corregido":>15}')
print('-' * 60)
print(f'{"Train Accuracy":<30} {history_saboteado["train_acc"][-1]:>14.2%} {history_corregido["train_acc"][-1]:>14.2%}')
print(f'{"Test Accuracy":<30} {history_saboteado["test_acc"][-1]:>14.2%} {history_corregido["test_acc"][-1]:>14.2%}')

gap_sab = history_saboteado['train_acc'][-1] - history_saboteado['test_acc'][-1]
gap_corr = history_corregido['train_acc'][-1] - history_corregido['test_acc'][-1]
print(f'{"Gap (Train-Test)":<30} {gap_sab:>14.2%} {gap_corr:>14.2%}')
print(f'{"Train Loss":<30} {history_saboteado["train_loss"][-1]:>15.4f} {history_corregido["train_loss"][-1]:>15.4f}')
print(f'{"Test Loss":<30} {history_saboteado["test_loss"][-1]:>15.4f} {history_corregido["test_loss"][-1]:>15.4f}')
print(f'{"Parámetros":<30} {n_params:>15,} {n_params_corr:>15,}')
print(f'{"Weight Decay":<30} {"0.0 (OFF)":>15} {"0.01 (ON)":>15}')
print(f'{"Dropout":<30} {"NO":>15} {"0.3":>15}')
print('=' * 60)

print(f'\n\n🎯 CONCLUSIONES:')
print(f'  1. DIAGNÓSTICO: El modelo memorizaba el dataset de entrenamiento')
print(f'     (Train ~100%, Test muy bajo). Overfitting violento.')
print(f'  2. INTERRUPTOR: Se activó Weight Decay (L2=0.01) y Dropout (p=0.3).')
print(f'     Se redujo la capacidad de la red.')
print(f'  3. RESULTADO: La brecha train-test se redujo de {gap_sab:.2%} a {gap_corr:.2%}.')
print(f'     El modelo ahora GENERALIZA en vez de memorizar.')

## 8. Bonus Matemático: Efecto de L2 en el Gradiente

Sin Weight Decay:
$$\theta_{t+1} = \theta_t - \eta \nabla L(\theta_t)$$

Con Weight Decay (L2):
$$\theta_{t+1} = \theta_t - \eta (\nabla L(\theta_t) + \lambda \theta_t)$$
$$= (1 - \eta\lambda)\theta_t - \eta \nabla L(\theta_t)$$

El término $(1 - \eta\lambda)$ **encoge los pesos** en cada paso, penalizando pesos grandes que memorizan ruido.

Con Dropout: En cada forward pass, se anulan neuronas al azar con probabilidad $p$:
$$h_i = \begin{cases} 0 & \text{con probabilidad } p \\ \frac{x_i}{1-p} & \text{con probabilidad } 1-p \end{cases}$$

Esto fuerza a la red a **no depender de neuronas individuales**, distribuyendo el aprendizaje.